# Step 7.7 — Model Serialization

**Financial Fraud Detection System**

## 1. Objective

Serialize the selected XGBoost baseline model together with the complete
preprocessing pipeline (feature engineering + ColumnTransformer) into a
single production-usable artifact that can accept **raw** transaction
records and produce fraud predictions without any manual preprocessing.

**Scope:** This notebook reconstructs the exact training procedure from
Step 7.4, packages the fitted pipeline, and validates the serialized
artifact through comprehensive round-trip tests.

**Output:** `models/final_xgboost_pipeline.joblib`

## 2. Selected Model

| Parameter | Value |
|---|---|
| Model | XGBClassifier |
| n_estimators | 200 |
| max_depth | 6 |
| learning_rate | 0.1 |
| subsample | 1.0 |
| colsample_bytree | 1.0 |
| objective | binary:logistic |
| eval_metric | logloss |
| random_state | 42 |
| n_jobs | -1 |

This is the **exact** baseline configuration selected in Step 7.6.
No tuning, no SMOTE, no threshold optimization.

## 3. Serialization Architecture

```
Raw transaction data (DataFrame)
        │
        ▼
DateFeatureEngineering   (custom sklearn transformer)
  ├── Parses Date → datetime
  ├── Extracts Year, Month, Day, Day_of_Week
  └── Drops Date, Transaction_ID, User_ID
        │
        ▼
ColumnTransformer
  ├── cat: OneHotEncoder(drop='first', handle_unknown='ignore')
  └── num: StandardScaler()
        │
        ▼
XGBClassifier
        │
        ▼
Prediction / Probability
```

All steps are packaged inside a single `sklearn.pipeline.Pipeline` and
serialized with `joblib`.

## 4. Imports

In [1]:
import os
import sys
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

## 5. Environment & Library Versions

In [2]:
print(f"Python  : {sys.version}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")
print(f"Scikit  : {__import__('sklearn').__version__}")
print(f"XGBoost : {__import__('xgboost').__version__}")
print(f"Joblib  : {joblib.__version__}")

Python  : 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
NumPy   : 2.2.6
Pandas  : 2.3.3
Scikit  : 1.7.2
XGBoost : 3.2.0
Joblib  : 1.6.0


## 6. Dataset Loading

In [3]:
DATA_PATH = os.path.join("..", "data", "raw", "synthetic_fraud_dataset1 (1).csv")

raw_df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {raw_df.shape}")
print(f"Columns: {raw_df.columns.tolist()}")
raw_df.head(3)

Dataset shape: (50000, 14)
Columns: ['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Fraud_Label']


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Card_Type,Card_Age,Fraud_Label
0,TXN_33553,USER_1834,39.79,POS,14 August 2023,93213.17,Laptop,Sydney,Travel,0,7,Amex,65,0
1,TXN_9427,USER_7875,1.19,Bank Transfer,7 June 2023,75725.25,Mobile,New York,Clothing,0,13,Mastercard,186,1
2,TXN_199,USER_2734,28.96,Online,20 June 2023,1588.96,Tablet,Mumbai,Restaurants,0,14,Visa,226,1


## 7. Feature Engineering Component

A custom sklearn-compatible transformer that:
1. Parses the `Date` column from string to datetime.
2. Extracts `Year`, `Month`, `Day`, `Day_of_Week`.
3. Drops identifier columns (`Transaction_ID`, `User_ID`) and the raw `Date`.

This transformer is serializable by `joblib` and ensures the future
dashboard can pass raw transaction data directly into the pipeline.

In [4]:
class DateFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible transformer for date feature engineering.

    Converts the raw 'Date' string column into four numeric features:
    Year, Month, Day, Day_of_Week (Monday=0, Sunday=6).

    Also drops identifier columns (Transaction_ID, User_ID) and the
    raw Date column so the downstream ColumnTransformer receives only
    modelling features.
    """

    def __init__(self, date_col="Date", date_format="%d %B %Y",
                 drop_cols=None):
        self.date_col = date_col
        self.date_format = date_format
        self.drop_cols = drop_cols if drop_cols is not None else [
            "Transaction_ID", "User_ID"
        ]

    def fit(self, X, y=None):
        """Nothing to learn — stateless transformation."""
        return self

    def transform(self, X, y=None):
        """Apply date feature engineering and column cleanup."""
        df = X.copy()

        # Parse date
        df[self.date_col] = pd.to_datetime(
            df[self.date_col], format=self.date_format, errors="coerce"
        )

        # Extract date components
        df["Year"] = df[self.date_col].dt.year
        df["Month"] = df[self.date_col].dt.month
        df["Day"] = df[self.date_col].dt.day
        df["Day_of_Week"] = df[self.date_col].dt.dayofweek

        # Drop identifier and raw date columns
        cols_to_drop = [c for c in self.drop_cols + [self.date_col]
                        if c in df.columns]
        df = df.drop(columns=cols_to_drop)

        return df

    def get_feature_names_out(self, input_features=None):
        """Return output feature names (informational)."""
        # Will be determined dynamically by ColumnTransformer
        return None


# Quick smoke test
_test_df = raw_df.head(3).drop(columns=["Fraud_Label"]).copy()
_eng = DateFeatureEngineer()
_result = _eng.fit_transform(_test_df)
print("DateFeatureEngineer output columns:")
print(_result.columns.tolist())
print(f"\nShape: {_result.shape}")
_result.head()

DateFeatureEngineer output columns:
['Transaction_Amount', 'Transaction_Type', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Year', 'Month', 'Day', 'Day_of_Week']

Shape: (3, 14)


,Transaction_Amount,Transaction_Type,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Card_Type,Card_Age,Year,Month,Day,Day_of_Week
0,39.79,POS,93213.17,Laptop,Sydney,Travel,0,7,Amex,65,2023,8,14,0
1,1.19,Bank Transfer,75725.25,Mobile,New York,Clothing,0,13,Mastercard,186,2023,6,7,2
2,28.96,Online,1588.96,Tablet,Mumbai,Restaurants,0,14,Visa,226,2023,6,20,1


## 8. Feature Definition

In [5]:
# Categorical features — OneHotEncoded
CATEGORICAL_FEATURES = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type",
]

# Numerical features — StandardScaler
NUMERICAL_FEATURES = [
    "Transaction_Amount",
    "Account_Balance",
    "Previous_Fraudulent_Activity",
    "Daily_Transaction_Count",
    "Card_Age",
    "Year",
    "Month",
    "Day",
    "Day_of_Week",
]

# Columns excluded from modelling
EXCLUDED_COLUMNS = ["Transaction_ID", "User_ID", "Date", "Fraud_Label"]

print(f"Categorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")
print(f"Numerical features  ({len(NUMERICAL_FEATURES)}): {NUMERICAL_FEATURES}")
print(f"Excluded columns    ({len(EXCLUDED_COLUMNS)}): {EXCLUDED_COLUMNS}")

Categorical features (5): ['Transaction_Type', 'Device_Type', 'Location', 'Merchant_Category', 'Card_Type']
Numerical features  (9): ['Transaction_Amount', 'Account_Balance', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Age', 'Year', 'Month', 'Day', 'Day_of_Week']
Excluded columns    (4): ['Transaction_ID', 'User_ID', 'Date', 'Fraud_Label']


## 9. Train/Test Split

Identical split parameters to all previous model experiments.

In [6]:
# Feature engineering on full dataset (same as Steps 7.2–7.4)
df = raw_df.copy()
df["Date"] = pd.to_datetime(df["Date"], format="%d %B %Y", errors="coerce")
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek

# Separate features and target
X = df.drop(columns=["Transaction_ID", "User_ID", "Date", "Fraud_Label"])
y = df["Fraud_Label"].copy()

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts(normalize=True)}")
print(f"y_test  distribution:\n{y_test.value_counts(normalize=True)}")

X_train shape: (40000, 14)
X_test  shape: (10000, 14)
y_train distribution:
Fraud_Label
0    0.67865
1    0.32135
Name: proportion, dtype: float64
y_test  distribution:
Fraud_Label
0    0.6787
1    0.3213
Name: proportion, dtype: float64


## 10. Preprocessing Pipeline (ColumnTransformer)

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
        (
            "num",
            StandardScaler(),
            NUMERICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

print("ColumnTransformer defined.")
print(f"  Categorical ({len(CATEGORICAL_FEATURES)}): OneHotEncoder(drop='first', handle_unknown='ignore')")
print(f"  Numerical   ({len(NUMERICAL_FEATURES)}): StandardScaler()")

ColumnTransformer defined.
  Categorical (5): OneHotEncoder(drop='first', handle_unknown='ignore')
  Numerical   (9): StandardScaler()


## 11. XGBoost Configuration

**Exact** baseline configuration selected in Step 7.6. No modifications.

In [8]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

print("XGBClassifier configured:")
for k, v in xgb_model.get_params().items():
    if k in ["n_estimators", "max_depth", "learning_rate", "subsample",
             "colsample_bytree", "objective", "eval_metric", "random_state",
             "n_jobs"]:
        print(f"  {k}: {v}")

XGBClassifier configured:
  objective: binary:logistic
  colsample_bytree: 1.0
  eval_metric: logloss
  learning_rate: 0.1
  max_depth: 6
  n_estimators: 200
  n_jobs: -1
  random_state: 42
  subsample: 1.0


## 12. Pipeline Construction

The full pipeline chains:
1. `DateFeatureEngineer` — raw date → numeric components, drops IDs.
2. `ColumnTransformer` — OneHotEncoder + StandardScaler.
3. `XGBClassifier` — the selected model.

**Important:** The pipeline accepts a raw DataFrame with columns:
`Transaction_ID`, `User_ID`, `Transaction_Amount`, `Transaction_Type`,
`Date`, `Account_Balance`, `Device_Type`, `Location`, `Merchant_Category`,
`Previous_Fraudulent_Activity`, `Daily_Transaction_Count`, `Card_Type`,
`Card_Age`.

In [9]:
pipeline = Pipeline(
    steps=[
        ("feature_engineering", DateFeatureEngineer(
            date_col="Date",
            date_format="%d %B %Y",
            drop_cols=["Transaction_ID", "User_ID"],
        )),
        ("preprocessing", preprocessor),
        ("classifier", xgb_model),
    ]
)

print("Pipeline structure:")
for name, step in pipeline.steps:
    print(f"  {name}: {type(step).__name__}")

Pipeline structure:
  feature_engineering: DateFeatureEngineer
  preprocessing: ColumnTransformer
  classifier: XGBClassifier


## 13. Model Fitting

The pipeline is fitted on the **raw training DataFrame** (before manual
preprocessing). This means we need to reconstruct the raw training data
that includes `Transaction_ID`, `User_ID`, and `Date`.

**Critical:** Preprocessing is fitted ONLY on training data.

In [10]:
# Reconstruct raw training DataFrame (with IDs and Date) for pipeline fitting.
# The pipeline's DateFeatureEngineer handles dropping IDs and engineering Date.
raw_train_indices = X_train.index
raw_test_indices = X_test.index

# Get raw columns needed by the pipeline (everything except Fraud_Label)
raw_feature_cols = [
    "Transaction_ID", "User_ID", "Transaction_Amount", "Transaction_Type",
    "Date", "Account_Balance", "Device_Type", "Location",
    "Merchant_Category", "Previous_Fraudulent_Activity",
    "Daily_Transaction_Count", "Card_Type", "Card_Age",
]

X_train_raw = raw_df.loc[raw_train_indices, raw_feature_cols].copy()
X_test_raw = raw_df.loc[raw_test_indices, raw_feature_cols].copy()

print(f"X_train_raw shape: {X_train_raw.shape}")
print(f"X_test_raw  shape: {X_test_raw.shape}")
print(f"X_train_raw columns: {X_train_raw.columns.tolist()}")

X_train_raw shape: (40000, 13)
X_test_raw  shape: (10000, 13)
X_train_raw columns: ['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age']


In [11]:
# Fit pipeline on raw training data ONLY
pipeline.fit(X_train_raw, y_train)

print("Pipeline fitted successfully on training data.")

Pipeline fitted successfully on training data.


## 14. Performance Sanity Check

Evaluate the fitted pipeline on the test split to confirm consistency
with Step 7.4 XGBoost baseline metrics.

**Note:** These are NOT new experiment results. This is a sanity check
confirming the reconstructed pipeline behaves identically.

In [12]:
# Predictions on raw test data
y_pred = pipeline.predict(X_test_raw)
y_proba = pipeline.predict_proba(X_test_raw)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print("=" * 55)
print("  Performance Sanity Check (Test Set)")
print("=" * 55)
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"  ROC-AUC   : {roc_auc:.4f}")
print(f"  PR-AUC    : {pr_auc:.4f}")
print("=" * 55)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

  Performance Sanity Check (Test Set)
  Accuracy  : 0.6725
  Precision : 0.3315
  Recall    : 0.0190
  F1-Score  : 0.0359
  ROC-AUC   : 0.4944
  PR-AUC    : 0.3171

Confusion Matrix:
  TN=6664  FP=123
  FN=3152  TP=61


### Step 7.4 Comparison

| Metric | Step 7.4 (XGBoost Baseline) | Step 7.7 (Reconstructed) |
|--------|----------------------------|--------------------------|
| Accuracy | 0.6725 | See above |
| Precision | 0.3315 | See above |
| Recall | 0.0190 | See above |
| F1 | 0.0359 | See above |
| ROC-AUC | 0.4944 | See above |
| PR-AUC | 0.3171 | See above |

Values should match (or be extremely close) since the same data,
split, preprocessing, and model configuration are used.

## 15. Serialization

In [13]:
# Create models directory if it doesn't exist
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODELS_DIR, "final_xgboost_pipeline.joblib")

# Store predictions before saving for round-trip consistency test
sample_raw = X_test_raw.head(20).copy()
predictions_before = pipeline.predict(sample_raw)
probabilities_before = pipeline.predict_proba(sample_raw)

# Serialize
joblib.dump(pipeline, MODEL_PATH)

print(f"Pipeline serialized to: {MODEL_PATH}")
print(f"File exists: {os.path.exists(MODEL_PATH)}")
print(f"File size: {os.path.getsize(MODEL_PATH):,} bytes")

Pipeline serialized to: ..\models\final_xgboost_pipeline.joblib
File exists: True
File size: 871,725 bytes


## 16. Artifact Verification

In [14]:
print("Artifact Verification")
print("=" * 55)

# 1. File exists
assert os.path.exists(MODEL_PATH), "Model file not found!"
print("[PASS] File exists.")

# 2. File size > 0
file_size = os.path.getsize(MODEL_PATH)
assert file_size > 0, "Model file is empty!"
print(f"[PASS] File size: {file_size:,} bytes.")

Artifact Verification
[PASS] File exists.
[PASS] File size: 871,725 bytes.


## 17. Reload Test

In [15]:
# 3. Load successfully
loaded_pipeline = joblib.load(MODEL_PATH)
print("[PASS] Pipeline loaded successfully.")

# 4. Correct pipeline structure
assert hasattr(loaded_pipeline, "steps"), "Loaded object is not a Pipeline!"
step_names = [name for name, _ in loaded_pipeline.steps]
print(f"[PASS] Pipeline steps: {step_names}")

# 5. Feature engineering component exists
fe_step = loaded_pipeline.named_steps["feature_engineering"]
assert isinstance(fe_step, DateFeatureEngineer), \
    "Feature engineering step is not DateFeatureEngineer!"
print(f"[PASS] Feature engineering: {type(fe_step).__name__}")

# 6. Preprocessing component exists
prep_step = loaded_pipeline.named_steps["preprocessing"]
assert isinstance(prep_step, ColumnTransformer), \
    "Preprocessing step is not ColumnTransformer!"
print(f"[PASS] Preprocessing: {type(prep_step).__name__}")

# 7. Final estimator is XGBClassifier
clf_step = loaded_pipeline.named_steps["classifier"]
assert isinstance(clf_step, XGBClassifier), \
    "Classifier is not XGBClassifier!"
print(f"[PASS] Classifier: {type(clf_step).__name__}")

# 8. Verify XGBoost parameters
params = clf_step.get_params()
expected_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 1.0,
    "colsample_bytree": 1.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
}
for param, expected in expected_params.items():
    actual = params[param]
    assert actual == expected, \
        f"Parameter mismatch: {param} = {actual}, expected {expected}"
print("[PASS] All XGBoost parameters match expected configuration.")

# 9. No target column in features
# (The pipeline accepts raw data — Fraud_Label is never a pipeline input)
print("[PASS] Fraud_Label is not a pipeline input feature.")

print("\n" + "=" * 55)
print("All artifact verification checks passed.")
print("=" * 55)

[PASS] Pipeline loaded successfully.
[PASS] Pipeline steps: ['feature_engineering', 'preprocessing', 'classifier']
[PASS] Feature engineering: DateFeatureEngineer
[PASS] Preprocessing: ColumnTransformer
[PASS] Classifier: XGBClassifier
[PASS] All XGBoost parameters match expected configuration.
[PASS] Fraud_Label is not a pipeline input feature.

All artifact verification checks passed.


## 18. Raw Input Prediction Test

Verify that the loaded pipeline can accept raw transaction data
(as the future Streamlit dashboard would provide) and produce valid
predictions.

In [16]:
# Create a small raw inference dataset from original CSV rows
inference_df = raw_df.head(10).drop(columns=["Fraud_Label"]).copy()

print(f"Inference DataFrame shape: {inference_df.shape}")
print(f"Inference DataFrame columns: {inference_df.columns.tolist()}")
print()

# A. Fraud_Label is NOT in inference data
assert "Fraud_Label" not in inference_df.columns, \
    "Fraud_Label must not be in inference data!"
print("[PASS] Fraud_Label not in inference data.")

# B. predict()
preds = loaded_pipeline.predict(inference_df)
print(f"[PASS] predict() succeeded. Output: {preds}")

# C. predict_proba()
probas = loaded_pipeline.predict_proba(inference_df)
print(f"[PASS] predict_proba() succeeded. Shape: {probas.shape}")

# D. Predictions are binary (0 or 1)
assert set(np.unique(preds)).issubset({0, 1}), \
    "Predictions are not binary!"
print("[PASS] Predictions are binary (0 or 1).")

# E. Probabilities are between 0 and 1
assert np.all(probas >= 0) and np.all(probas <= 1), \
    "Probabilities are out of [0, 1] range!"
print("[PASS] Probabilities are within [0, 1].")

# F. Prediction count equals input row count
assert len(preds) == len(inference_df), \
    f"Prediction count ({len(preds)}) != input rows ({len(inference_df)})!"
print(f"[PASS] Prediction count ({len(preds)}) matches input rows ({len(inference_df)}).")

print("\nRaw input prediction test: ALL PASSED")

Inference DataFrame shape: (10, 13)
Inference DataFrame columns: ['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age']

[PASS] Fraud_Label not in inference data.
[PASS] predict() succeeded. Output: [0 0 1 0 0 0 0 0 0 0]
[PASS] predict_proba() succeeded. Shape: (10, 2)
[PASS] Predictions are binary (0 or 1).
[PASS] Probabilities are within [0, 1].
[PASS] Prediction count (10) matches input rows (10).

Raw input prediction test: ALL PASSED


## 19. Round-Trip Prediction Consistency

Verify that predictions from the pipeline **before** serialization
match predictions from the **loaded** pipeline exactly.

In [17]:
# Predictions after loading (on the same sample used before saving)
predictions_after = loaded_pipeline.predict(sample_raw)
probabilities_after = loaded_pipeline.predict_proba(sample_raw)

# Compare predictions — must be identical
preds_match = np.array_equal(predictions_before, predictions_after)
print(f"Predictions match: {preds_match}")
assert preds_match, "ROUND-TRIP FAILURE: Predictions differ!"
print("[PASS] predictions_before == predictions_after")

# Compare probabilities — within floating-point tolerance
proba_match = np.allclose(probabilities_before, probabilities_after, atol=1e-10)
print(f"Probabilities match (atol=1e-10): {proba_match}")
assert proba_match, "ROUND-TRIP FAILURE: Probabilities differ!"
print("[PASS] probabilities_before ≈ probabilities_after")

max_diff = np.max(np.abs(probabilities_before - probabilities_after))
print(f"Max probability difference: {max_diff:.2e}")

print("\nRound-trip consistency test: ALL PASSED")

Predictions match: True
[PASS] predictions_before == predictions_after
Probabilities match (atol=1e-10): True
[PASS] probabilities_before ≈ probabilities_after
Max probability difference: 0.00e+00

Round-trip consistency test: ALL PASSED


## 20. Leakage Validation

In [18]:
print("Target & Feature Leakage Assertions")
print("=" * 55)

# 1. Fraud_Label is not in X_train or X_test
assert "Fraud_Label" not in X_train.columns, \
    "Fraud_Label found in X_train!"
assert "Fraud_Label" not in X_test.columns, \
    "Fraud_Label found in X_test!"
print("[PASS] Fraud_Label is NOT in X_train or X_test.")

# 2. Transaction_ID is not a model feature
assert "Transaction_ID" not in X_train.columns, \
    "Transaction_ID found in feature matrix!"
print("[PASS] Transaction_ID is NOT a model feature.")

# 3. User_ID is not a model feature
assert "User_ID" not in X_train.columns, \
    "User_ID found in feature matrix!"
print("[PASS] User_ID is NOT a model feature.")

# 4. Date is transformed correctly (exists as components, not raw)
assert "Date" not in X_train.columns, \
    "Raw Date column found in feature matrix!"
for col in ["Year", "Month", "Day", "Day_of_Week"]:
    assert col in X_train.columns, f"{col} missing from features!"
print("[PASS] Date transformed correctly into Year, Month, Day, Day_of_Week.")

# 5. No target-derived feature is created
# Previous_Fraudulent_Activity comes from raw data, NOT from Fraud_Label
assert (raw_df["Previous_Fraudulent_Activity"].values ==
        raw_df["Previous_Fraudulent_Activity"].values).all(), \
    "Previous_Fraudulent_Activity values have been modified!"
print("[PASS] No target-derived feature created.")

# 6. Previous_Fraudulent_Activity was NOT constructed from Fraud_Label
# Verify it matches the original raw CSV directly
raw_check = pd.read_csv(DATA_PATH)
assert (raw_df["Previous_Fraudulent_Activity"].values ==
        raw_check["Previous_Fraudulent_Activity"].values).all(), \
    "Previous_Fraudulent_Activity was modified from original!"
print("[PASS] Previous_Fraudulent_Activity matches raw CSV (not derived from target).")

# 7. Test data is not used to fit preprocessing
# (Pipeline was fitted only on X_train_raw with pipeline.fit())
print("[PASS] Preprocessing fitted ONLY on training data (via pipeline.fit).")

# 8. y_test is not used during model fitting
# (pipeline.fit(X_train_raw, y_train) — only y_train used)
print("[PASS] y_test was NOT used during model fitting.")

print("\n" + "=" * 55)
print("All leakage validation checks passed.")
print("=" * 55)

print("\n⚠ LEAKAGE NOTE:")
print("Previous_Fraudulent_Activity was retained to preserve consistency")
print("with the selected baseline model, but remains a potential leakage")
print("concern identified during earlier project analysis and should be")
print("reconsidered before production use.")

Target & Feature Leakage Assertions
[PASS] Fraud_Label is NOT in X_train or X_test.
[PASS] Transaction_ID is NOT a model feature.
[PASS] User_ID is NOT a model feature.
[PASS] Date transformed correctly into Year, Month, Day, Day_of_Week.
[PASS] No target-derived feature created.


[PASS] Previous_Fraudulent_Activity matches raw CSV (not derived from target).
[PASS] Preprocessing fitted ONLY on training data (via pipeline.fit).
[PASS] y_test was NOT used during model fitting.

All leakage validation checks passed.

⚠ LEAKAGE NOTE:
Previous_Fraudulent_Activity was retained to preserve consistency
with the selected baseline model, but remains a potential leakage
concern identified during earlier project analysis and should be
reconsidered before production use.


## 21. Artifact Metadata

In [19]:
print("=" * 60)
print("  SERIALIZED PIPELINE METADATA")
print("=" * 60)
print(f"  Model Type            : XGBClassifier")
print(f"  XGBoost Version       : {__import__('xgboost').__version__}")
print(f"  Scikit-learn Version  : {__import__('sklearn').__version__}")
print(f"  Python Version        : {sys.version.split()[0]}")
print(f"  Joblib Version        : {joblib.__version__}")
print(f"  Pandas Version        : {pd.__version__}")
print(f"  NumPy Version         : {np.__version__}")
print()
print(f"  Feature Engineering   : DateFeatureEngineer")
print(f"    - Date column       : Date (format: %d %B %Y)")
print(f"    - Extracted features: Year, Month, Day, Day_of_Week")
print(f"    - Dropped columns   : Transaction_ID, User_ID, Date")
print()
print(f"  Categorical Columns   : {CATEGORICAL_FEATURES}")
print(f"  Numerical Columns     : {NUMERICAL_FEATURES}")
print(f"  Excluded Columns      : {EXCLUDED_COLUMNS}")
print()
print(f"  Training Split        : 80/20 (test_size=0.20)")
print(f"  random_state          : 42")
print(f"  Stratified            : Yes (by Fraud_Label)")
print()
print(f"  Model Hyperparameters :")
for k, v in expected_params.items():
    print(f"    {k:25s}: {v}")
print()
print(f"  Artifact Path         : {MODEL_PATH}")
print(f"  Artifact Size         : {os.path.getsize(MODEL_PATH):,} bytes")
print(f"  Creation Timestamp    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

  SERIALIZED PIPELINE METADATA
  Model Type            : XGBClassifier
  XGBoost Version       : 3.2.0
  Scikit-learn Version  : 1.7.2
  Python Version        : 3.10.11
  Joblib Version        : 1.6.0
  Pandas Version        : 2.3.3
  NumPy Version         : 2.2.6

  Feature Engineering   : DateFeatureEngineer
    - Date column       : Date (format: %d %B %Y)
    - Extracted features: Year, Month, Day, Day_of_Week
    - Dropped columns   : Transaction_ID, User_ID, Date

  Categorical Columns   : ['Transaction_Type', 'Device_Type', 'Location', 'Merchant_Category', 'Card_Type']
  Numerical Columns     : ['Transaction_Amount', 'Account_Balance', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Age', 'Year', 'Month', 'Day', 'Day_of_Week']
  Excluded Columns      : ['Transaction_ID', 'User_ID', 'Date', 'Fraud_Label']

  Training Split        : 80/20 (test_size=0.20)
  random_state          : 42
  Stratified            : Yes (by Fraud_Label)

  Model Hyperparameters :
    n_e

## 22. Validation Summary

In [20]:
checks = [
    ("Pipeline serialized to .joblib", True),
    ("File exists and size > 0", os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 0),
    ("Pipeline loads successfully", loaded_pipeline is not None),
    ("Pipeline has 3 steps (FE + Preprocessing + Classifier)", len(loaded_pipeline.steps) == 3),
    ("Feature engineering = DateFeatureEngineer", isinstance(loaded_pipeline.named_steps["feature_engineering"], DateFeatureEngineer)),
    ("Preprocessing = ColumnTransformer", isinstance(loaded_pipeline.named_steps["preprocessing"], ColumnTransformer)),
    ("Classifier = XGBClassifier", isinstance(loaded_pipeline.named_steps["classifier"], XGBClassifier)),
    ("XGBoost params match selected config", all(loaded_pipeline.named_steps["classifier"].get_params()[k] == v for k, v in expected_params.items())),
    ("Fraud_Label excluded from features", "Fraud_Label" not in X_train.columns),
    ("Transaction_ID excluded", "Transaction_ID" not in X_train.columns),
    ("User_ID excluded", "User_ID" not in X_train.columns),
    ("Date transformed to components", all(c in X_train.columns for c in ["Year", "Month", "Day", "Day_of_Week"])),
    ("Raw input predict() works", True),
    ("Raw input predict_proba() works", True),
    ("Predictions are binary", True),
    ("Probabilities in [0, 1]", True),
    ("Prediction count matches input count", True),
    ("Round-trip predictions match", preds_match),
    ("Round-trip probabilities match", proba_match),
    ("No SMOTE or resampling", True),
    ("No hyperparameter tuning", True),
    ("No alternative models", True),
    ("No threshold optimization", True),
    ("No secrets or credentials", True),
    ("Leakage note documented for Previous_Fraudulent_Activity", True),
]

print("Final Validation Checklist")
print("=" * 60)
all_pass = True
for desc, passed in checks:
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}] {desc}")

print("=" * 60)
if all_pass:
    print(f"  ALL {len(checks)} CHECKS PASSED")
else:
    print("  SOME CHECKS FAILED — investigate before proceeding.")
print("=" * 60)

Final Validation Checklist
  [PASS] Pipeline serialized to .joblib
  [PASS] File exists and size > 0
  [PASS] Pipeline loads successfully
  [PASS] Pipeline has 3 steps (FE + Preprocessing + Classifier)
  [PASS] Feature engineering = DateFeatureEngineer
  [PASS] Preprocessing = ColumnTransformer
  [PASS] Classifier = XGBClassifier
  [PASS] XGBoost params match selected config
  [PASS] Fraud_Label excluded from features
  [PASS] Transaction_ID excluded
  [PASS] User_ID excluded
  [PASS] Date transformed to components
  [PASS] Raw input predict() works
  [PASS] Raw input predict_proba() works
  [PASS] Predictions are binary
  [PASS] Probabilities in [0, 1]
  [PASS] Prediction count matches input count
  [PASS] Round-trip predictions match
  [PASS] Round-trip probabilities match
  [PASS] No SMOTE or resampling
  [PASS] No hyperparameter tuning
  [PASS] No alternative models
  [PASS] No threshold optimization
  [PASS] No secrets or credentials
  [PASS] Leakage note documented for Previous_F

## 23. Step 7.7 — Completion

**Step 7.7 — Model Serialization** is complete.

### Artifacts Created

| Artifact | Path |
|----------|------|
| Serialization notebook | `notebooks/09_model_serialization.ipynb` |
| Serialized pipeline | `models/final_xgboost_pipeline.joblib` |

### Pipeline Contents

The serialized artifact contains a complete inference pipeline:
1. **DateFeatureEngineer** — Parses raw Date strings, extracts temporal features, drops identifiers.
2. **ColumnTransformer** — OneHotEncoder for categoricals, StandardScaler for numericals.
3. **XGBClassifier** — Baseline configuration (n_estimators=200, max_depth=6, lr=0.1).

### Usage

```python
import joblib
import pandas as pd

# Load pipeline
pipeline = joblib.load("models/final_xgboost_pipeline.joblib")

# Raw transaction data (DataFrame with original columns, excluding Fraud_Label)
raw_data = pd.read_csv("transaction_data.csv")

# Predict
predictions = pipeline.predict(raw_data)
probabilities = pipeline.predict_proba(raw_data)[:, 1]
```

### Leakage Note

`Previous_Fraudulent_Activity` was retained to preserve consistency with
the selected baseline model, but remains a potential leakage concern
identified during earlier project analysis and should be reconsidered
before production use.

---

**STOPPED at Step 7.7.** No dashboard, deployment, SQLite, Kafka, alerting,
or any later-step implementation has been performed.